# EXIST 2026: Multimodal sexism detection in memes

Trains and evaluates the two fusion architectures (cross-attention and gated fusion) on text, image and the viewers' physiological signals (EEG, eye tracking, heart rate), then predicts the official test set and writes the submission files.

Subtasks: **2.1** sexism identification (YES/NO) · **2.2** source intention (DIRECT/JUDGEMENTAL) · **2.3** sexism categorization (multi-label).

A GPU is recommended (Google Colab works).

In [ ]:
import pandas as pd

from src.data import build_splits, load_memes
from src.submission import apply_hierarchy, write_submission
from src.train import FINAL_FUSION, predict, train_task

DATASET_DIR = "data/EXIST 2026 Memes Dataset"
TRAIN_DIR = f"{DATASET_DIR}/training"
TEST_DIR = f"{DATASET_DIR}/test"
EPOCHS = None  # None uses each architecture's default (5 for cross-attention, 3 for gated fusion)

## 1. Data
One hard label per meme by majority vote of the annotators (union of categories for 2.3). Memes without consensus are dropped. 80/20 train/dev split per language.

In [ ]:
splits = build_splits(load_memes(f"{TRAIN_DIR}/EXIST2026_training.json"))

pd.DataFrame(
    [{"lang": lang, "task": task, "train": len(train), "dev": len(dev)} for (lang, task), (train, dev) in splits.items()]
)

## 2. Compare fusion strategies
English uses XLM-RoBERTa and Spanish uses RoBERTuito as text encoder; images go through ViT.

In [ ]:
trained = {}
for (lang, task), (train, dev) in splits.items():
    for fusion in ("cross_attention", "gated"):
        trained[(lang, task, fusion)] = train_task(train, dev, lang, task, fusion, TRAIN_DIR, epochs=EPOCHS)

results = pd.DataFrame(
    [{"lang": lang, "task": task, "fusion": fusion, "dev_f1": model.history[-1]["dev_f1"]}
     for (lang, task, fusion), model in trained.items()]
)
results.pivot_table(index="fusion", columns=["task", "lang"], values="dev_f1").round(3)

## 3. Predict the test set with the selected models
Gated fusion for 2.1 and 2.3, cross-attention for 2.2.

In [ ]:
test = load_memes(f"{TEST_DIR}/EXIST2026_test_clean.json")

predictions = []
for lang in ("en", "es"):
    memes = test[test["lang"] == lang][["id_EXIST", "text", "path_memes", "sensorial"]].copy()
    for task, fusion in FINAL_FUSION.items():
        memes[task] = predict(trained[(lang, task, fusion)], memes, TEST_DIR)
    predictions.append(memes)
predictions = pd.concat(predictions, ignore_index=True)

write_submission(apply_hierarchy(predictions), out_dir="submissions")